# Ingesting and preparing data to build the OMOP 5.3.1 Lakehouse


<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/hls/patient-readmission/hls-patient-readmision-flow-1.png" style="float: left; margin-right: 30px; margin-top:10px" width="650px" />

Our first step in reducing patient readmission risk is to ingest and prepare data from multiple external sources as simple tables our downstream Analysts can leverage. To do so, we will leverage the OMOP data model.

The OMOP database is a Common Data Model to harmonize data analysis required for medical product safety surveillance, comparative effectiveness, quality of care, and patient-level predictive modeling.

Databricks is uniquely positioned to create a leverage this model: not only it provides capabilities to build, ingest and transform the data, but it also let your Data Analyst and Data Scientist team to run advanced analysis on top of it.

<br style="clear: both">

## Implementing the OMOP model

<img src="https://ohdsi.github.io/TheBookOfOhdsi/images/CommonDataModel/cdmDiagram.png" width="400px" style="float: right; margin-left: 50px" >

The OMOP database contains several tables.

- Clinical Data Tables (CDT)
- Health System Data Tables (HSDT)
- Health Economics Data Tables (HEDT)
- Standardized Derived Elements (SDE)
- Metadata Tables (MT)
- Vocabulary Tables (VT)

In this demo, we will consume raw data artificially generated from Synthea and apply transformations to translate them as OMOP schema.

*To keep this demo simple, we will only implement a subste of the OMOP data model*

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F01-Data-Ingestion%2F01.1-SDP-patient-readmission&demo_name=lakehouse-hls-readmission&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-hls-readmission%2F01-Data-Ingestion%2F01.1-SDP-patient-readmission&version=1">

## Spark Declarative Pipelines: A simple way to build and manage data pipelines for fresh, high quality data!


In this notebook, we'll work as a Data Engineer to build our OMOP CDM database. <br>
We'll consume and clean our raw data sources to prepare the tables required for our BI & ML workload.

The lakehouse makes it easy to ingest from any external or internal HLS datasources, such as HL7 FHIR, EHR, ioMT, SQL databases). For our demo, our data will be files received in a cloud blob storage in different format. We will then apply a couple of transformations while ensuring data quality.

Databricks simplifies this task with Spark Declarative Pipelines (SDP) by making Data Engineering accessible to all.

SDP allows Data Analysts to create advanced pipeline with plain SQL.

<div>
  <div style="width: 45%; float: left; margin-bottom: 10px; padding-right: 45px">
    <p>
      <img style="width: 50px; float: left; margin: 0px 5px 30px 0px;" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/logo-accelerate.png"/> 
      <strong>Accelerate ETL development</strong> <br/>
      Enable analysts and data engineers to innovate rapidly with simple pipeline development and maintenance 
    </p>
    <p>
      <img style="width: 50px; float: left; margin: 0px 5px 30px 0px;" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/logo-complexity.png"/> 
      <strong>Remove operational complexity</strong> <br/>
      By automating complex administrative tasks and gaining broader visibility into pipeline operations
    </p>
  </div>
  <div style="width: 48%; float: left">
    <p>
      <img style="width: 50px; float: left; margin: 0px 5px 30px 0px;" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/logo-trust.png"/> 
      <strong>Trust your data</strong> <br/>
      With built-in quality controls and quality monitoring to ensure accurate and useful BI, Data Science, and ML 
    </p>
    <p>
      <img style="width: 50px; float: left; margin: 0px 5px 30px 0px;" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/logo-stream.png"/> 
      <strong>Simplify batch and streaming</strong> <br/>
      With self-optimization and auto-scaling data pipelines for batch or streaming processing 
    </p>
</div>
</div>

<br style="clear:both">

<img src="https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-logo.png" style="float: right;" width="200px">




The SDP pipeline was started for you! Click to <a dbdemos-pipeline-id="sdp-patient-readmission" href="#joblist/pipelines/35207ea8-0eb4-47bf-a925-46e95bc7af9f" target="_blank">open your Pipeline</a> and review its execution.


## Building a Spark Declarative Pipelines pipeline to ingest and prepare HLS data with the OMOP model

In this example, we'll implement an end-to-end SDP pipeline consuming the aforementioned information. We'll use the medaillon architecture but we could build star schema, data vault, or any other modelisation.

We'll incrementally load new data with the autoloader, clean and enrich this information.

This information will then be used to build our cohorts and build SQL dashboard to analyze our patients.

Let's implement the following flow: 

 <img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/hls/patient-readmission/hls-patient-readmision-dlt-0.png" width="1000px" />

### 1/ Data Exploration
All Data projects start with some exploration. Open the [explorations/sample_exploration]($./explorations/sample_exploration) notebook to get started and discover the data made available to you



### 2/ Loading our data using Databricks Autoloader (read_files)

<img width="650px" style="float:right" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/hls/patient-readmission/hls-patient-readmision-dlt-1.png"/>

  
Our raw files are available in our landing zone as CSV. We need to ingest them at scale, while handling schema inference and evolution.
Databricks Autoloader makes this very easy.

For more details on autoloader, run `dbdemos.install('auto-loader')`

Open the [transformations/01-bronze.sql]($./transformations/01-bronze.sql) notebook to review the SQL queries ingesting the raw data and creating our bronze layer.


### 3/ Enforce quality and materialize our tables for Data Analysts

<img width="650px" style="float:right" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/hls/patient-readmission/hls-patient-readmision-dlt-2.png"/>

The next layer often call silver is consuming **incremental** data from the bronze one, and cleaning up some information.

We're also adding an TODO [expectation](https://docs.databricks.com/workflows/delta-live-tables/delta-live-tables-expectations.html) on different field to enforce and track our Data Quality. This will ensure that our dashboard are relevant and easily spot potential errors due to data anomaly.

For more advanced SDP capabilities run `dbdemos.install('pipeline-bike')` or `dbdemos.install('declarative-pipeline-cdc')` for CDC/SCDT2 example.

These tables are clean and ready to be used by the BI team!



Open the [transformations/02-silver.sql]($./transformations/02-silver.sql) notebook to review the SQL queries creating our features and our training dataset

### 4/ Final tables for our Data Analysis and ML model

<img width="650px" style="float:right" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/hls/patient-readmission/hls-patient-readmision-dlt-3.png"/>

Finally, let's cbuild our final tables containing clean data that we'll be able to use of to build our cohorts and predict patient risks.

These tables are clean and ready to be used by the BI team!




Open the [transformations/03-gold.sql]($./transformations/03-gold.sql) notebook to review the SQL queries creating our features and our training dataset

## Our pipeline is now ready!

As you can see, building Data Pipeline with databricks let you focus on your business implementation while the engine solves all hard data engineering work for you.

Now that these tables are available in our Lakehouse, let's review how we can share them with the Data Scientists and Data Analysts teams.

Open the <a dbdemos-pipeline-id="sdp-patient-readmission" href="#joblist/pipelines/35207ea8-0eb4-47bf-a925-46e95bc7af9f" target="_blank">OMOP data model Spark Declarative Pipelines pipeline</a> and click on start to visualize your lineage and consume the new data incrementally!

# Next: secure and share data with Unity Catalog

Now that these tables are available in our Lakehouse, let's review how we can share them with the Data Scientists and Data Analysts teams.

Jump to the [Governance with Unity Catalog notebook]($../02-Data-Governance/02-Data-Governance-patient-readmission) or [Go back to the introduction]($../00-patient-readmission-introduction)